# SkinAI — DenseNet121 분류 모델 학습

**Google Colab / 로컬 환경 모두 지원**
- Colab: Drive 마운트 → 레포 클론 → 심링크
- 로컬: 현재 디렉토리 그대로 사용

In [19]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    # Colab 런타임 임시 경로 (세션 종료 시 소실)
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    # 로컬: 노트북이 위치한 레포 루트 그대로 사용
    LOCAL_ROOT = str(Path.cwd())
    PROJECT_ROOT = LOCAL_ROOT

print(f"환경       : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경       : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [20]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Drive 마운트 경로 (Colab 런타임 전용)
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    if not Path(COLAB_ROOT).exists():
        !git clone https://github.com/kyoe-23/skin_ai.git {COLAB_ROOT}
    else:
        # 이미 존재하면 최신 코드로 업데이트
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")

Already up to date.


In [22]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 경로 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    # Colab 런타임: Drive에 있는 dataset_14를 심링크로 연결
    os.makedirs("data", exist_ok=True)
    DRIVE_DATASET = f"{DRIVE_ROOT}/data/dataset_14"
    if Path(DRIVE_DATASET).exists():
        !ln -sfn {DRIVE_DATASET} data/dataset_14
        print("data/dataset_14 심링크 생성 완료")
    else:
        print(f"경고: Drive 데이터셋 없음 — {DRIVE_DATASET}")
else:
    # 로컬: data/dataset_14 존재 여부만 확인
    if not Path("data/dataset_14").exists():
        print("경고: data/dataset_14/ 없음 — AI Hub 데이터를 먼저 배치하세요")
    else:
        print("data/dataset_14/ 확인 완료")

!ls data/

현재 디렉토리: /content/colab_skin_ai
data/dataset_14 심링크 생성 완료
dataset_14  processed


In [23]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

# Grad-CAM (추론 서버 사용 시)
# !pip install -q pytorch-grad-cam

In [32]:
# ── 셀 6-1: 학습 실행 ─────────────────────────────────────────────
# CSV의 zip_path(로컬 절대경로)를 --root_dir로 실행 환경 경로에 맞게 재매핑

# DenseNet121 (기본)
!python -m ai.training.classifier.train \
    --backbone densenet121 \
    --num_epochs 100 \
    --batch_size 256 \
    --resume ai/results/epoch_75.pth \
    --root_dir {PROJECT_ROOT}

# EfficientNet-B3 (비교)
# !python -m ai.training.classifier.train \
#     --backbone efficientnet_b3 \
#     --num_epochs 100 \
#     --batch_size 256 \
#     --root_dir {PROJECT_ROOT}

# 세션 만료 후 재개
# !python -m ai.training.classifier.train \
#    --backbone densenet121 \
#    --num_epochs 100 \
#    --batch_size 256 \
#    --resume ai/results/epoch_N.pth \
#    --root_dir {PROJECT_ROOT}


INFO [INFO] CUDA 사용
AI Hub 08-14 분류 모델 학습
  backbone : densenet121
  device   : cuda
  epochs   : 100
  batch    : 256
  lr       : 0.001
  warmup   : 3 epochs
  scheduler: CosineAnnealingLR (T_max=97)
INFO Dataset 로드: data/processed/train.csv (4800건, direction=front)
INFO Dataset 로드: data/processed/val.csv (600건, direction=front)

  Train: 4800건
  Val  : 600건
INFO   Model     : DenseNet
INFO   Total     : 6,960,006 params
INFO   Trainable : 6,960,006 params

  이어서 학습: epoch 75, best_top1=0.8037

학습 시작...

[Epoch 76/100] Train Loss: 0.009 | Train Top-1: 99.7%
                    Val   Loss: 1.184 | Val   Top-1: 75.5% | Val Top-3: 97.3%
                    시간: 78.9s | LR: 0.000473

[Epoch 77/100] Train Loss: 0.008 | Train Top-1: 99.7%
                    Val   Loss: 1.174 | Val   Top-1: 75.9% | Val Top-3: 98.2%
                    시간: 79.7s | LR: 0.000472

[Epoch 78/100] Train Loss: 0.010 | Train Top-1: 99.7%
                    Val   Loss: 1.068 | Val   Top-1: 76.4% | Val Top-3: 97.2%


In [41]:
# ── 셀 6-2 ─────────────────────────────────────────────
import json
from pathlib import Path

# 두 경로의 체크포인트를 합쳐서 표시 (경로 변경 이력 반영)
search_dirs = ["ai/results", "ai/checkpoints/aihub"]
all_epoch_ckpts = sorted(
    {p.name: p for d in search_dirs for p in Path(d).glob("epoch_*.pth") if Path(d).exists()}.values(),
    key=lambda p: int(p.stem.split("_")[1])
)
print("주기 체크포인트:")
for p in all_epoch_ckpts:
    print(f"  {p.name}  ({p.parent})")

log_path = next(
    (Path(d) / "training_log.json" for d in search_dirs if (Path(d) / "training_log.json").exists()),
    None,
)
if log_path:
    with open(log_path) as f:
        log = json.load(f)
    last = log["history"][-1]
    print(f"\n마지막 에폭: {last['epoch']}  |  val_top1: {last['val_top1']*100:.2f}%")
    print(f"best 에폭:   {log['best_epoch']}  |  best_top1: {log['best_val_top1']*100:.2f}%")


주기 체크포인트:
  epoch_5.pth  (ai/checkpoints/aihub)
  epoch_10.pth  (ai/checkpoints/aihub)
  epoch_15.pth  (ai/checkpoints/aihub)
  epoch_20.pth  (ai/checkpoints/aihub)
  epoch_25.pth  (ai/checkpoints/aihub)
  epoch_30.pth  (ai/checkpoints/aihub)
  epoch_35.pth  (ai/checkpoints/aihub)
  epoch_40.pth  (ai/checkpoints/aihub)
  epoch_45.pth  (ai/checkpoints/aihub)
  epoch_50.pth  (ai/results)
  epoch_55.pth  (ai/results)
  epoch_60.pth  (ai/results)
  epoch_65.pth  (ai/results)
  epoch_70.pth  (ai/results)
  epoch_75.pth  (ai/results)
  epoch_80.pth  (ai/results)
  epoch_85.pth  (ai/results)
  epoch_90.pth  (ai/results)
  epoch_95.pth  (ai/results)
  epoch_100.pth  (ai/results)

마지막 에폭: 100  |  val_top1: 79.91%
best 에폭:   99  |  best_top1: 80.79%


In [42]:
# ── 셀 7: 평가 + Threshold 최적화 ──────────────────────────────
!python -m ai.testing.evaluate \
    --checkpoint ai/results/best.pth \
    --root_dir {PROJECT_ROOT}

!python -m ai.testing.threshold_opt \
    --checkpoint ai/results/best.pth \
    --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/val.csv (600건, direction=front)
평가 데이터: val.csv (600건)

 SkinAI 분류 모델 평가 결과 (AI Hub 08-14)
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 80.50%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 97.33%
 Macro F1-Score : 0.8096
 Macro AUC      : 0.9583
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.7905   0.8300   0.8098   0.9605
 아토피피부염         0.8454   0.8200   0.8325   0.9798
 여드름            0.8667   0.6500   0.7429   0.9640
 주사             0.9157   0.7600   0.8306   0.9460
 지루피부염          0.5500   0.7700   0.6417   0.8997
 정상             1.0000   1.0000   1.0000   1.0000
/content/colab_skin_ai/ai/testing/evaluate.py:115: UserWarning: Glyph 44148 (\N{HANGUL SYLLABLE GEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/colab_skin_ai/ai/test

In [37]:
# ── 셀 8: 체크포인트 Drive 저장 (Colab 전용) ────────────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from pathlib import Path

    # 실제 존재하는 체크포인트 경로 자동 감지
    CANDIDATE_DIRS = [
        f"{COLAB_ROOT}/ai/results",
        f"{COLAB_ROOT}/ai/checkpoints/aihub",
    ]
    CKPT_SRC = next((d for d in CANDIDATE_DIRS if Path(d).exists()), None)
    if CKPT_SRC is None:
        raise FileNotFoundError("체크포인트 디렉토리를 찾을 수 없습니다.")

    CKPT_DST = f"{DRIVE_ROOT}/ai/results"
    shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
    print(f"Drive 저장 완료: {CKPT_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/ 에 저장됨")

Drive 저장 완료: /content/drive/MyDrive/skin_ai/ai/results
